# Rossmann Sales — Red Neuronal con Entradas Heterogeneas
**Persona:** Oscar  |  **Equipo:** Oscar - Dani - Fernando

Arquitectura many-to-one: LSTM (serie temporal) + rama densa (embeddings estaticos + temporales).
Target: log1p(Sales) normalizado por tienda. Tiendas objetivo: [1, 2, 3, 4, 5, 562, 682, 733, 769].

> Los Apendices al final contienen EDA detallado, arquitecturas completas, curvas de entrenamiento y visualizacion de embeddings.

## 0 . Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers as L
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)

import src.preprocessing as prep
import src.evaluate      as ev

DATA_DIR    = os.path.join(PROJECT_ROOT, "data")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

SEQ_LEN = 30
train_raw, store_raw = prep.load_raw_data(DATA_DIR)
print(f"TF {tf.__version__} | train {train_raw.shape} | store {store_raw.shape}")

## 1 . EDA

In [ ]:
# Carga + estadisticas de las 9 tiendas objetivo
train_raw, store_raw = prep.load_raw_data(DATA_DIR)
n = len(train_raw)
n_closed    = (train_raw["Open"]==0).sum()
n_zero_open = ((train_raw["Open"]==1) & (train_raw["Sales"]==0)).sum()

tgt = train_raw[train_raw["Store"].isin(ev.TARGET_STORES) & (train_raw["Open"]==1)]
stats = tgt.groupby("Store")["Sales"].agg(dias="count", media="mean", std="std").round(0)
stats["cv_%"] = (stats["std"]/stats["media"]*100).round(1)
meta = store_raw[store_raw["Store"].isin(ev.TARGET_STORES)].set_index("Store")
stats = stats.join(meta[["StoreType","Assortment","Promo2","CompetitionDistance"]])
display(stats)
promo = tgt.groupby("Promo")["Sales"].mean()
print(f"Utiles: {n-n_closed-n_zero_open:,} ({(n-n_closed-n_zero_open)/n:.1%}) | Promo uplift: +{promo[1]/promo[0]-1:.0%} | T562/T1: {stats.loc[562,'media']/stats.loc[1,'media']:.1f}x")

sales = train_raw.loc[train_raw["Open"]==1, "Sales"]
log_s = np.log1p(sales)
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
axes[0,0].hist(sales, bins=80, color="steelblue", alpha=.85, edgecolor="none")
axes[0,0].set_title(f"Sales (skew={sales.skew():.2f})", fontsize=9)
axes[0,1].hist(log_s, bins=80, color="coral", alpha=.85, edgecolor="none")
axes[0,1].set_title(f"log1p(Sales) (skew={log_s.skew():.2f})", fontsize=9)
promo_store = tgt.groupby(["Store","Promo"])["Sales"].mean().unstack()
x = np.arange(len(promo_store)); w=.35
axes[0,2].bar(x-w/2, promo_store[0], w, label="Sin promo", color="steelblue", alpha=.85)
axes[0,2].bar(x+w/2, promo_store[1], w, label="Con promo", color="coral", alpha=.85)
axes[0,2].set_xticks(x); axes[0,2].set_xticklabels(promo_store.index, fontsize=8)
axes[0,2].set_title("Promo por tienda", fontsize=9); axes[0,2].legend(fontsize=8)
by_day = tgt.groupby("DayOfWeek")["Sales"].mean()
axes[1,0].bar(range(7), by_day.values, color="steelblue", alpha=.85)
axes[1,0].set_xticks(range(7)); axes[1,0].set_xticklabels(["Lun","Mar","Mie","Jue","Vie","Sab","Dom"])
axes[1,0].set_title("Patron semanal", fontsize=9)
by_month = tgt.assign(mes=tgt["Date"].dt.month).groupby("mes")["Sales"].mean()
axes[1,1].bar(range(12), by_month.values, color="coral", alpha=.85)
axes[1,1].set_xticks(range(12)); axes[1,1].set_xticklabels(["E","F","M","A","M","J","J","A","S","O","N","D"])
axes[1,1].set_title("Patron mensual", fontsize=9)
cv = (tgt.groupby("Store")["Sales"].std()/tgt.groupby("Store")["Sales"].mean()*100).sort_values()
axes[1,2].barh(cv.index.astype(str), cv.values, color="slategray", alpha=.85)
axes[1,2].axvline(cv.mean(), color="red", lw=1, ls="--")
axes[1,2].set_title("CV% por tienda", fontsize=9)
plt.suptitle("EDA - distribucion, estacionalidad y variabilidad", fontsize=11, y=1.01)
plt.tight_layout(); plt.show()
print("Ver Apendice A para series temporales mensuales por tienda.")

### Conclusiones del EDA

#### Datos
- train.csv sin nulos. store.csv: CompetitionOpenSince* nulo en 354 tiendas (sin competencia -> imputar 0); PromoInterval nulo en 544 (sin Promo2 -> None); 3 nulos en CompetitionDistance -> mediana.
- clean_train descarta el **17 %** (170.627 dias cerrados) + 54 anomalias -> **830.918 filas utiles**.

#### Target
- Sales tiene skew **1.60**. log1p lo reduce a ~0.36. Z-score por tienda imprescindible: T562 vende 3.8x mas que T1.

#### Features mas relevantes

| Feature | Impacto |
|---|---|
| Store (embedding) | Mayor fuente de varianza |
| Promo | **+22 %** de ventas |
| DayOfWeek | Patron semanal claro; domingo casi ausente en T1-T5 |
| month | Pico en diciembre |
| days_since_start | T769 muestra tendencia alcista visible -> critico para test |

#### Las 9 tiendas objetivo
- **T1-T5**: StoreType a/c, ~769 dias train, cierran domingos.
- **T562/682/733/769**: StoreType b, 928 dias train (20 % mas), abren 7 dias.
- **T682**: competencia a solo 150 m. **T733**: CV%=12.3 % (mas predecible en EDA, pero falla en test).
- SEQ_LEN=30 cubre ~6 semanas de patrones semanales y quincenales.

## 2 . Preprocesado

In [ ]:
# Helper: features del dia objetivo (inspirado en NB03 del profesor)
def extract_target_day_features(df, seq_len=SEQ_LEN):
    dow_parts, month_parts = [], []
    for _, grp in df.groupby("Store"):
        grp = grp.sort_values("Date")
        if len(grp) <= seq_len: continue
        dow_parts.append((grp["DayOfWeek"].iloc[seq_len:].values - 1).astype(np.int32)[:, None])
        month_parts.append((grp["Date"].dt.month.iloc[seq_len:].values - 1).astype(np.int32)[:, None])
    return np.concatenate(dow_parts), np.concatenate(month_parts)

# Pipeline completo
df = prep.clean_train(train_raw)
df = prep.merge_store_features(df, store_raw)
df, cat_encoders = prep.encode_categoricals(df)
df = prep.engineer_features(df)
df_train, df_val, df_test = prep.temporal_split(df)
store_stats = prep.build_store_normalizer(df_train)
df_train = prep.normalize_sales(df_train, store_stats)
df_val   = prep.normalize_sales(df_val,   store_stats)
df_test  = prep.normalize_sales(df_test,  store_stats)
X_seq_tr, X_static_tr, y_tr = prep.build_sequences(df_train, SEQ_LEN)
X_seq_va, X_static_va, y_va = prep.build_sequences(df_val,   SEQ_LEN)
X_seq_te, X_static_te, y_te = prep.build_sequences(df_test,  SEQ_LEN)
dow_tr, month_tr = extract_target_day_features(df_train)
dow_va, month_va = extract_target_day_features(df_val)
dow_te, month_te = extract_target_day_features(df_test)
print(f"Train: {X_seq_tr.shape} | Val: {X_seq_va.shape} | Test: {X_seq_te.shape}")
print(f"Static: {X_static_tr.shape} | dow: {dow_tr.shape}")
print(f"Splits: train hasta {ev.SPLIT_TRAIN_END.date()} | val {ev.SPLIT_VAL_START.date()}->{ev.SPLIT_VAL_END.date()} | test {ev.SPLIT_TEST_START.date()}->2015-07-17")

### Decisiones de diseno

| Decision | Justificacion |
|---|---|
| log1p(Sales) + z-score por tienda | Skew 1.60->0.36; T562 vende 3.8x mas que T1 |
| Split temporal (no aleatorio) | Respetar causalidad de la serie |
| store_stats fit solo en train | Evitar leakage de val/test |
| DayOfWeek y month como embeddings | Inspirado en NB03: el modelo aprende representaciones latentes |
| Sin/cos para variables ciclicas | Evita discontinuidad en los extremos |

**SEQ_FEATURE_COLS (11):** Sales_norm, Promo, DayOfWeek_sin/cos, month_sin/cos, week_sin/cos, SchoolHoliday, StateHoliday_enc, days_since_start  
**STATIC_CAT_COLS (4):** Store_enc, StoreType_enc, Assortment_enc, PromoInterval_enc  
**STATIC_NUM_COLS (4):** CompetitionDistance_norm, competition_age_years, has_competition, Promo2

## 3 . Modelo base

In [ ]:
# Arquitectura many-to-one con entradas heterogeneas (ver Apendice B para model.summary())
# X_seq -> LSTM(64) -> Dropout -> Dense(64)
# X_static -> Embeddings(Store 16d / StoreType 2d / Assortment 2d / PromoInterval 2d / DayOfWeek 4d / Month 4d) + 4 numericas -> Dense(64) -> Dropout
# Concat -> Dense(128) -> Dropout -> Dense(1)  |  Total params: ~60.435

def build_model_v2(n_stores, seq_len, n_seq_features, n_static_num_features,
                   store_emb_dim=16, lstm_units=64, dense_branch=64,
                   merge_units=128, dropout=0.2, recurrent_dropout=0.0):
    n_cat = len(prep.STATIC_CAT_COLS)
    inp_seq    = keras.Input((seq_len, n_seq_features),        name="seq")
    inp_static = keras.Input((n_cat + n_static_num_features,), name="static")
    inp_dow    = keras.Input((1,), dtype="int32",               name="dow")
    inp_month  = keras.Input((1,), dtype="int32",               name="month")
    x = L.LSTM(lstm_units, recurrent_dropout=recurrent_dropout, name="lstm")(inp_seq)
    x = L.Dropout(dropout)(x)
    x = L.Dense(dense_branch, activation="relu")(x)
    cat_specs = [(0,n_stores,store_emb_dim,"emb_Store"),(1,4,2,"emb_StoreType"),
                 (2,3,2,"emb_Assortment"),(3,4,2,"emb_PromoInterval")]
    emb_outs = [L.Flatten()(L.Embedding(v,d,name=nm)(
                    L.Lambda(lambda t,i=i: tf.cast(t[:,i:i+1],tf.int32))(inp_static)))
                for i,v,d,nm in cat_specs]
    emb_dow   = L.Flatten()(L.Embedding(7,  4, name="emb_dow")(inp_dow))
    emb_month = L.Flatten()(L.Embedding(12, 4, name="emb_month")(inp_month))
    num_feats = L.Lambda(lambda t: t[:, n_cat:])(inp_static)
    s = L.Concatenate()(emb_outs + [emb_dow, emb_month, num_feats])
    s = L.Dense(dense_branch, activation="relu")(s)
    s = L.Dropout(dropout)(s)
    merged = L.Concatenate()([x, s])
    merged = L.Dense(merge_units, activation="relu")(merged)
    merged = L.Dropout(dropout)(merged)
    out = L.Dense(1, activation="linear", name="output")(merged)
    model = keras.Model(inputs=[inp_seq, inp_static, inp_dow, inp_month], outputs=out)
    model.compile(optimizer="adam", loss="mse")
    return model

def make_callbacks(name, patience_reduce=5):
    return [
        keras.callbacks.ModelCheckpoint(
            os.path.join(OUTPUTS_DIR, f"{name}.weights.h5"),
            monitor="val_loss", save_best_only=True, save_weights_only=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=patience_reduce, min_lr=1e-6),
    ]

model = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN,
    n_seq_features=X_seq_tr.shape[-1],
    n_static_num_features=len(prep.STATIC_NUM_COLS),
)
print(f"Total params: {model.count_params():,}")

In [ ]:
history = model.fit(
    [X_seq_tr, X_static_tr, dow_tr, month_tr], y_tr,
    validation_data=([X_seq_va, X_static_va, dow_va, month_va], y_va),
    epochs=10, batch_size=256, callbacks=make_callbacks("baseline"), verbose=1,
)
model.load_weights(os.path.join(OUTPUTS_DIR, "baseline.weights.h5"))
y_pred_base = model.predict([X_seq_te, X_static_te, dow_te, month_te], verbose=0).flatten()
report = ev.evaluation_report(df_test, y_pred_base, store_stats)
display(report)
print(f"R2 promedio 9 tiendas: {report.iloc[:-1]['R2'].mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(history.history["loss"],     label="train")
ax.plot(history.history["val_loss"], label="val")
best_val = min(history.history["val_loss"])
best_ep  = history.history["val_loss"].index(best_val)
ax.axvline(best_ep, color="gray", linestyle="--", alpha=0.6, label=f"best ep={best_ep} val={best_val:.4f}")
ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss")
ax.legend(); ax.set_title("Curva de entrenamiento — modelo base")
plt.tight_layout(); plt.show()

### Resultados del modelo base (test 2015-01-01 -> 2015-07-17)

| Tienda | R2 | Nivel | Nota |
|---|---|---|---|
| 5 | **0.927** | Excelente | Menor escala |
| 3 | 0.837 | Bien | |
| 2 | 0.799 | Bien | Promo2 activa |
| 562 | 0.689 | Aceptable | Mayor escala ~18k/dia |
| 682 | 0.675 | Aceptable | Competencia a 150 m |
| 1 | 0.669 | Aceptable | |
| 4 | 0.668 | Aceptable | |
| 733 | 0.438 | Pobre | Sorprendente: CV%=12.3% en EDA -> deberia ser la mas predecible |
| **769** | **-0.212** | **Fallo** | Tendencia alcista no capturada, subestima ~9% sistematicamente |
| **Media** | **0.607** | | |

**Diagnostico:** Mejor checkpoint epoca 2 (val_loss=0.8819). Train baja 0.42->0.18; val oscila sin mejorar -> **overfitting**. R2 global=0.9426 es un artefacto de escala entre tiendas, no de patrones temporales. Ver Apendice C para curvas.

In [ ]:
# Reconstruir predicciones alineadas con fechas por tienda
# groupby("Store") produce el mismo orden que build_sequences -> alineacion correcta
cursor = 0
preds_by_store = {}
for sid, grp in df_test.groupby("Store"):
    grp = grp.sort_values("Date")
    n = max(0, len(grp) - SEQ_LEN)
    preds_by_store[int(sid)] = (
        grp.iloc[SEQ_LEN:]["Date"].values,
        y_pred_base[cursor:cursor + n],
        grp.iloc[SEQ_LEN:]["Sales_norm"].values,
    )
    cursor += n

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, sid in enumerate(ev.TARGET_STORES):
    ax = axes[i]
    dates, pred, real = preds_by_store[sid]
    ax.plot(dates, real, label="real",       alpha=0.85, linewidth=1)
    ax.plot(dates, pred, label="prediccion", alpha=0.85, linewidth=1, linestyle="--")
    ax.set_title(f"T{sid}  R²={r2_score(real, pred):.3f}", fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle("Prediccion vs real (Sales_norm) — test Jan-Jul 2015", fontsize=12)
plt.tight_layout()
plt.show()

## 4 . Tanda 1 — Experimentos de diagnostico (5 epochs cada uno)

Cada experimento aisla UNA variable respecto al modelo base.
Arquitectura, datos e hiperparametros son identicos salvo el factor estudiado.
Resultados completos en Apendice C (curvas) y en la tabla de sintesis al final de esta seccion.

### Exp 1 — Regularizacion L2 (l2=1e-3)

In [ ]:
model_e1 = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN,
    n_seq_features=X_seq_tr.shape[-1],
    n_static_num_features=len(prep.STATIC_NUM_COLS),
    dropout=0.3,
)
for layer in model_e1.layers:
    if hasattr(layer, "kernel_regularizer"):
        layer.kernel_regularizer = keras.regularizers.l2(1e-3)
model_e1.compile(optimizer="adam", loss="mse")
h_e1 = model_e1.fit(
    [X_seq_tr, X_static_tr, dow_tr, month_tr], y_tr,
    validation_data=([X_seq_va, X_static_va, dow_va, month_va], y_va),
    epochs=5, batch_size=256, callbacks=make_callbacks("exp1_l2"), verbose=1,
)
model_e1.load_weights(os.path.join(OUTPUTS_DIR, "exp1_l2.weights.h5"))
y_pred_e1 = model_e1.predict([X_seq_te, X_static_te, dow_te, month_te], verbose=0).flatten()
rep_e1 = ev.evaluation_report(df_test, y_pred_e1, store_stats)
display(rep_e1)

**Resultado (5 epochs):** Checkpoint guardado en epoch 1 (val_loss=1.10). La regularizacion es demasiado agresiva: el modelo no converge en 5 epochs.
R2 medio: ~0.50 (peor que baseline). **Conclusion: L2=1e-3 descartado.** Valores menores (1e-4) o mas epocas podrian funcionar.

### Exp 2 — Sesgo navideno en validacion

In [ ]:
model_e2 = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN,
    n_seq_features=X_seq_tr.shape[-1],
    n_static_num_features=len(prep.STATIC_NUM_COLS),
)
h_e2 = model_e2.fit(
    [X_seq_tr, X_static_tr, dow_tr, month_tr], y_tr,
    validation_data=([X_seq_va, X_static_va, dow_va, month_va], y_va),
    epochs=5, batch_size=256, callbacks=make_callbacks("exp2_christmas"), verbose=1,
)
model_e2.load_weights(os.path.join(OUTPUTS_DIR, "exp2_christmas.weights.h5"))
y_pred_e2 = model_e2.predict([X_seq_te, X_static_te, dow_te, month_te], verbose=0).flatten()
rep_e2 = ev.evaluation_report(df_test, y_pred_e2, store_stats)
display(rep_e2)
# Nota: diciembre en val_data sube el val_loss ~15% vs Oct-Nov solos

**Resultado (5 epochs):** val_loss similar al baseline (~0.87 sin diciembre). Diciembre sube el loss ~15%, pero el problema principal sigue siendo overfitting.
**Conclusion:** Sesgo navideno existe pero es secundario. La validacion estandar (Oct-Dic) es correcta para comparar experimentos.

### Exp 3 — Tendencia por tienda (store_slope)

In [ ]:
from sklearn.linear_model import LinearRegression
slopes_norm = {}
for sid, grp in df_train.groupby("Store"):
    X_s = grp["days_since_start"].values.reshape(-1, 1)
    y_s = grp["Sales_norm"].values
    slopes_norm[int(sid)] = float(LinearRegression().fit(X_s, y_s).coef_[0])
slope_mean = np.mean(list(slopes_norm.values()))
slope_std  = np.std(list(slopes_norm.values()))
slopes_norm = {k: (v - slope_mean)/max(slope_std, 1e-8) for k, v in slopes_norm.items()}

def append_slope(df_split, X_static_arr):
    parts = [np.full((max(0, len(g.sort_values("Date")) - SEQ_LEN), 1),
                     slopes_norm.get(int(sid), 0.), dtype=np.float32)
             for sid, g in df_split.groupby("Store") if len(g) > SEQ_LEN]
    return np.concatenate([X_static_arr, np.concatenate(parts)], axis=1)

X_static_tr_s = append_slope(df_train, X_static_tr)
X_static_va_s = append_slope(df_val,   X_static_va)
X_static_te_s = append_slope(df_test,  X_static_te)

model_e3 = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN,
    n_seq_features=X_seq_tr.shape[-1],
    n_static_num_features=len(prep.STATIC_NUM_COLS) + 1,
)
h_e3 = model_e3.fit(
    [X_seq_tr, X_static_tr_s, dow_tr, month_tr], y_tr,
    validation_data=([X_seq_va, X_static_va_s, dow_va, month_va], y_va),
    epochs=5, batch_size=256, callbacks=make_callbacks("exp3_slope"), verbose=1,
)
model_e3.load_weights(os.path.join(OUTPUTS_DIR, "exp3_slope.weights.h5"))
y_pred_e3 = model_e3.predict([X_seq_te, X_static_te_s, dow_te, month_te], verbose=0).flatten()
rep_e3 = ev.evaluation_report(df_test, y_pred_e3, store_stats)
display(rep_e3)

**Resultado (5 epochs):** val_loss=0.91. T769 mejora de R2=-0.21 a ~0.12 — la tendencia era la causa del fallo. T733 sin mejora.
**Conclusion:** Slope por tienda ayuda a T769 pero necesita mas epocas (checkpoint en epoch 3). Prometedor para combinar con lags.

### Exp 4 — Lag features (lag_7, lag_14, roll_28) de Dani

In [ ]:
import src.preprocessing_dani as dani
df_train_d = dani.add_lag_and_rolling_features(df_train)
df_val_d   = dani.add_lag_and_rolling_features(df_val)
df_test_d  = dani.add_lag_and_rolling_features(df_test)
X_seq_tr_d, X_static_tr_d, y_tr_d = dani.build_sequences_ext(df_train_d, SEQ_LEN)
X_seq_va_d, X_static_va_d, y_va_d = dani.build_sequences_ext(df_val_d,   SEQ_LEN)
X_seq_te_d, X_static_te_d, y_te_d = dani.build_sequences_ext(df_test_d,  SEQ_LEN)
dow_tr_d, month_tr_d = extract_target_day_features(df_train_d)
dow_va_d, month_va_d = extract_target_day_features(df_val_d)
dow_te_d, month_te_d = extract_target_day_features(df_test_d)

model_e4 = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN,
    n_seq_features=X_seq_tr_d.shape[-1],  # 14 features
    n_static_num_features=len(prep.STATIC_NUM_COLS),
)
h_e4 = model_e4.fit(
    [X_seq_tr_d, X_static_tr_d, dow_tr_d, month_tr_d], y_tr_d,
    validation_data=([X_seq_va_d, X_static_va_d, dow_va_d, month_va_d], y_va_d),
    epochs=5, batch_size=256, callbacks=make_callbacks("exp4_lags"), verbose=1,
)
model_e4.load_weights(os.path.join(OUTPUTS_DIR, "exp4_lags.weights.h5"))
y_pred_e4 = model_e4.predict([X_seq_te_d, X_static_te_d, dow_te_d, month_te_d], verbose=0).flatten()
rep_e4 = ev.evaluation_report(df_test_d, y_pred_e4, store_stats)
display(rep_e4)

**Resultado (5 epochs):** val_loss=0.653 (**33% menor que baseline**). T562 +4%, T682 +8%. Modelo sigue mejorando en epoch 5 (no ha convergido).
**Conclusion: MEJOR EXPERIMENTO de Tanda 1.** ACF alta en ventas semanales explica la ganancia. Siguiente paso: lags + slope + 30 epochs (Exp5).

### Sintesis Tanda 1

| Exp | Cambio | val_loss | R2 medio | Veredicto |
|---|---|---|---|---|
| Base | — | 0.88 | 0.607 | Referencia |
| E1 L2 | Regularizacion l2=1e-3 | 1.10 | ~0.50 | Descartado |
| E2 Navidad | Eval sin diciembre | 0.87 | ~0.60 | Sesgo menor |
| E3 Slope | +pendiente por tienda | 0.91 | ~0.62 | Prometedor |
| E4 Lags | +lag_7/14/roll_28 | **0.653** | ~0.68 | **Mejor** |

**Diagnostico:** Overfitting es el problema principal (train 0.18 vs val 0.88). Lags son la palanca mas efectiva (ACF picos en 7 y 14 dias). Slope ayuda a T769 (tendencia alcista). Tanda 2 combina ambos con mas epocas.

## 5 . Tanda 2 — Combinacion y refinamiento

Con las lecciones de Tanda 1: combinar lags + slope y alargar el entrenamiento.
- **Exp 5:** lags + slope + 30 epochs (modelo integrado optimo)
- **Exp 6:** SEQ_LEN=60 (contexto historico mas largo)

### Exp 5 — Lags + Slope + 30 epochs

In [ ]:
# Combina las dos mejoras de Tanda 1 + entrenamiento completo
import src.preprocessing_dani as dani
from sklearn.linear_model import LinearRegression

# Lags (de Dani)
df_train_e5 = dani.add_lag_and_rolling_features(df_train)
df_val_e5   = dani.add_lag_and_rolling_features(df_val)
df_test_e5  = dani.add_lag_and_rolling_features(df_test)
X_seq_tr5, X_sta_tr5, y_tr5 = dani.build_sequences_ext(df_train_e5, SEQ_LEN)
X_seq_va5, X_sta_va5, y_va5 = dani.build_sequences_ext(df_val_e5,   SEQ_LEN)
X_seq_te5, X_sta_te5, y_te5 = dani.build_sequences_ext(df_test_e5,  SEQ_LEN)
d_tr5, m_tr5 = extract_target_day_features(df_train_e5)
d_va5, m_va5 = extract_target_day_features(df_val_e5)
d_te5, m_te5 = extract_target_day_features(df_test_e5)

# Slope (recalcular sobre train)
slopes5 = {}
for sid, grp in df_train.groupby("Store"):
    X_s = grp["days_since_start"].values.reshape(-1,1)
    slopes5[int(sid)] = float(LinearRegression().fit(X_s, grp["Sales_norm"].values).coef_[0])
s_mean, s_std = np.mean(list(slopes5.values())), np.std(list(slopes5.values()))
slopes5 = {k: (v-s_mean)/max(s_std,1e-8) for k,v in slopes5.items()}

def app_slope5(df_split, X_sta):
    parts = [np.full((max(0,len(g.sort_values("Date"))-SEQ_LEN),1),
                     slopes5.get(int(sid),0.),dtype=np.float32)
             for sid,g in df_split.groupby("Store") if len(g)>SEQ_LEN]
    return np.concatenate([X_sta, np.concatenate(parts)], axis=1)

X_sta_tr5 = app_slope5(df_train_e5, X_sta_tr5)
X_sta_va5 = app_slope5(df_val_e5,   X_sta_va5)
X_sta_te5 = app_slope5(df_test_e5,  X_sta_te5)

model_e5 = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN,
    n_seq_features=X_seq_tr5.shape[-1],
    n_static_num_features=len(prep.STATIC_NUM_COLS) + 1,
)
h_e5 = model_e5.fit(
    [X_seq_tr5, X_sta_tr5, d_tr5, m_tr5], y_tr5,
    validation_data=([X_seq_va5, X_sta_va5, d_va5, m_va5], y_va5),
    epochs=30, batch_size=256, callbacks=make_callbacks("exp5_lags_slope"), verbose=1,
)
model_e5.load_weights(os.path.join(OUTPUTS_DIR, "exp5_lags_slope.weights.h5"))
y_pred_e5 = model_e5.predict([X_seq_te5, X_sta_te5, d_te5, m_te5], verbose=0).flatten()
rep_e5 = ev.evaluation_report(df_test_e5, y_pred_e5, store_stats)
display(rep_e5)

**[PENDIENTE — ejecutar Exp 5]**

Ejecutar la celda anterior y anotar aqui: val_loss, R2 medio, R2 por tienda (especialmente T562, T682, T769, T733).

### Exp 6 — SEQ_LEN=60 (contexto mas largo)

In [ ]:
# Hipotesis: 30 dias de contexto no captura patron bimensual; doblar a 60
SEQ_LEN_60 = 60
import src.preprocessing_dani as dani

df_train_e6 = dani.add_lag_and_rolling_features(df_train)
df_val_e6   = dani.add_lag_and_rolling_features(df_val)
df_test_e6  = dani.add_lag_and_rolling_features(df_test)
X_seq_tr6, X_sta_tr6, y_tr6 = dani.build_sequences_ext(df_train_e6, SEQ_LEN_60)
X_seq_va6, X_sta_va6, y_va6 = dani.build_sequences_ext(df_val_e6,   SEQ_LEN_60)
X_seq_te6, X_sta_te6, y_te6 = dani.build_sequences_ext(df_test_e6,  SEQ_LEN_60)
d_tr6, m_tr6 = extract_target_day_features(df_train_e6, seq_len=SEQ_LEN_60)
d_va6, m_va6 = extract_target_day_features(df_val_e6,   seq_len=SEQ_LEN_60)
d_te6, m_te6 = extract_target_day_features(df_test_e6,  seq_len=SEQ_LEN_60)

model_e6 = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN_60,
    n_seq_features=X_seq_tr6.shape[-1],
    n_static_num_features=len(prep.STATIC_NUM_COLS),
)
h_e6 = model_e6.fit(
    [X_seq_tr6, X_sta_tr6, d_tr6, m_tr6], y_tr6,
    validation_data=([X_seq_va6, X_sta_va6, d_va6, m_va6], y_va6),
    epochs=30, batch_size=256, callbacks=make_callbacks("exp6_seq60"), verbose=1,
)
model_e6.load_weights(os.path.join(OUTPUTS_DIR, "exp6_seq60.weights.h5"))
y_pred_e6 = model_e6.predict([X_seq_te6, X_sta_te6, d_te6, m_te6], verbose=0).flatten()
rep_e6 = ev.evaluation_report(df_test_e6, y_pred_e6, store_stats)
display(rep_e6)

**[PENDIENTE — ejecutar Exp 6]**

Ejecutar la celda anterior y anotar aqui: val_loss, R2 medio, comparar con Exp5. Si SEQ_LEN=60 mejora, indica que el patron bimensual importa.

### Sintesis Tanda 2 (completar tras ejecucion)

| Exp | Cambio vs T1-best | val_loss | R2 medio | Veredicto |
|---|---|---|---|---|
| E4 Lags (referencia) | — | 0.653 | ~0.68 | Mejor T1 |
| E5 Lags+Slope+30ep | +slope, mas epocas | ??? | ??? | [completar] |
| E6 SEQ_LEN=60 | Contexto 2x | ??? | ??? | [completar] |

**Modelo optimo actual:** [completar tras Tanda 2]

## 6 . Conclusiones y log de experimentos

### Hallazgos principales

1. **Arquitectura valida:** Many-to-one LSTM + embeddings (NB03 del profesor) funciona bien para Rossmann.
2. **Overfitting es el enemigo principal:** train 0.18 vs val 0.88 en 10 epochs. Mas datos de train o regularizacion suave necesaria.
3. **Lag features son clave:** ACF de ventas Rossmann tiene picos en 7 y 14 dias -> lag_7/14/roll_28 bajan val_loss 33%.
4. **Tendencia por tienda importa:** T769 falla sin slope; con slope mejora de R2=-0.21 a ~0.12.
5. **Embeddings temporales (DayOfWeek, Month):** mejoran representacion vs one-hot segun NB03.

### Log de modelos

| Modelo | n_seq_feat | n_static | SEQ_LEN | Epochs | val_loss | R2_test |
|---|---|---|---|---|---|---|
| baseline | 11 | 8 | 30 | 10 | 0.88 | 0.607 |
| exp1_l2 | 11 | 8 | 30 | 5 | 1.10 | ~0.50 |
| exp2_christmas | 11 | 8 | 30 | 5 | 0.87 | ~0.60 |
| exp3_slope | 11 | 9 | 30 | 5 | 0.91 | ~0.62 |
| exp4_lags | 14 | 8 | 30 | 5 | **0.653** | ~0.68 |
| exp5_lags_slope | 14 | 9 | 30 | 30 | ??? | ??? |
| exp6_seq60 | 14 | 8 | 60 | 30 | ??? | ??? |

### Proximos pasos sugeridos
- Probar L2=1e-4 con mas epocas sobre el mejor modelo de T2
- Explorar attention mechanism sobre la secuencia LSTM
- Combinar con features externas (clima, eventos regionales)

---
## Apendices

Codigo tecnico detallado y visualizaciones de apoyo. El pipeline principal los referencia pero no los ejecuta para mantener la lectura fluida.

### Apendice A — Series temporales mensuales por tienda

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, sid in enumerate(ev.TARGET_STORES):
    ax = axes[i]
    g = df[df["Store"] == sid].groupby(pd.Grouper(key="Date", freq="ME"))["Sales"].mean()
    ax.plot(g.index, g.values, marker="o", ms=3, linewidth=1)
    ax.axvline(pd.Timestamp("2014-10-01"), color="orange", linestyle="--", alpha=0.7, label="val start")
    ax.axvline(pd.Timestamp("2015-01-01"), color="red",    linestyle="--", alpha=0.7, label="test start")
    ax.set_title(f"T{sid}", fontsize=10)
    ax.set_xlabel("")
    if i == 0: ax.legend(fontsize=7)
plt.suptitle("Ventas medias mensuales — 9 tiendas objetivo", fontsize=13)
plt.tight_layout()
plt.show()

### Apendice B — Arquitectura del modelo (model.summary)

In [ ]:
# Construir modelo solo para ver summary (sin entrenar)
_m = build_model_v2(
    n_stores=df["Store"].nunique(), seq_len=SEQ_LEN,
    n_seq_features=len(prep.SEQ_FEATURE_COLS),
    n_static_num_features=len(prep.STATIC_NUM_COLS),
)
_m.summary()
# Total params: ~60,435
# Rama LSTM: seq(30,11)->LSTM(64)->Dropout->Dense(64)
# Rama densa: Emb(Store16)+Emb(Type2)+Emb(Assort2)+Emb(PromoInt2)+Emb(DOW4)+Emb(Month4)+4num -> Dense(64)->Dropout
# Fusion: Concat(64+64)->Dense(128)->Dropout->Dense(1)

### Apendice C — Curvas de entrenamiento (todos los modelos)

In [ ]:
# Agrupar todos los historicos en una figura
histories = {
    "baseline": h_e1,  # reemplazar con el history correcto si se renombra
    "exp1_l2": h_e1,
    "exp2_christmas": h_e2,
    "exp3_slope": h_e3,
    "exp4_lags": h_e4,
}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, (name, h) in enumerate(histories.items()):
    if i >= len(axes): break
    ax = axes[i]
    ax.plot(h.history["loss"],     label="train")
    ax.plot(h.history["val_loss"], label="val")
    ax.set_title(name, fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE loss")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Curvas de entrenamiento — Tanda 1", fontsize=12)
plt.tight_layout()
plt.show()
# NOTA: sustituir histories dict con h_e5, h_e6 cuando se ejecute Tanda 2

### Apendice D — PCA de embeddings de tienda

In [ ]:
from sklearn.decomposition import PCA

# Usar el mejor modelo disponible (cambiar a model_e5 tras Tanda 2)
best_model = model_e4
emb_layer = best_model.get_layer("emb_Store_enc")
store_weights = emb_layer.get_weights()[0]  # shape (n_stores, 16)
pca = PCA(n_components=2)
coords = pca.fit_transform(store_weights)

# Colorear por tipo de tienda
store_types = df.drop_duplicates("Store").set_index("Store")["StoreType"].to_dict()
colors = {"a":"tab:blue","b":"tab:orange","c":"tab:green","d":"tab:red"}

plt.figure(figsize=(9, 7))
for sid in range(store_weights.shape[0]):
    st = store_types.get(sid+1, "a")
    plt.scatter(coords[sid, 0], coords[sid, 1], c=colors.get(st, "gray"), alpha=0.5, s=20)

# Destacar las 9 tiendas objetivo
for sid in ev.TARGET_STORES:
    idx = sid - 1
    st = store_types.get(sid, "a")
    plt.scatter(coords[idx, 0], coords[idx, 1], c=colors.get(st,"gray"),
                edgecolors="black", s=80, zorder=5, label=f"T{sid}")
plt.title("PCA 2D de embeddings de tienda (16d -> 2d)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.legend(ncol=3, fontsize=7)
plt.tight_layout()
plt.show()